# 示例: 运行带闸门的河道水力模型

**脚本:** `examples/run_structure_model.py`

## 目的
此示例用于演示如何在`preissmann_model`中设置并运行一个包含闸门（Gate）的水力模型。闸门作为一种常见的水工建筑物，可以在河道中控制水位和流量。

此笔记本将展示如何：
1.  定义一个闸门结构，包括其位置、开度和宽度。
2.  将闸门添加到一个`HydraulicModel`实例中。
3.  运行一个模拟，观察闸门对上游水位的壅水效应。
4.  检查并可视化最终的水面线，以验证闸门的效果。

## 1. 环境设置

首先，我们导入所需的模型组件。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Gate

## 2. 定义河道和闸门

我们创建一个具有一定坡度的河道，并在其中间节点（节点10）处定义一个闸门。闸门的开度 `opening_height` 设为0.5米，这意味着水流只能从一个0.5米高的开口下流过。

In [ ]:
num_nodes = 21
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=10) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 250.0), # 5km total length
    slope=0.001,
    manning_n=0.03
)

# 定义闸门
gate = Gate(name="mid_reach_gate", node_index=10, opening_height=0.5, width=10)
print(f"定义闸门: '{gate.name}' @ 节点 {gate.node_index}")

## 3. 创建水力模型

我们创建一个`HydraulicModel`实例，并将上面创建的闸门作为`structures`参数传递进去。

In [ ]:
downstream_level = 2.0
river = HydraulicModel(
    name="R1_regulated_river",
    reach=reach_geom,
    dt=10.0,
    downstream_level=downstream_level,
    structures=[gate]
)

# 设置初始条件
river.Q[:] = 15.0
for i in range(num_nodes):
    river.Z[i] = river.Z_bed[i] + 2.5

print(f"创建水力模型: '{river.name}'")

## 4. 运行模拟

我们给定一个恒定的上游入流，运行一个足够长的模拟，让河道内的水流达到稳定状态。

In [ ]:
num_steps = 100
inflow_hydrograph = np.full(num_steps, 15.0)
downstream_hydrograph = np.full(num_steps, downstream_level)

print(f"运行模拟 {num_steps} 步...")
river.run(
    num_steps=num_steps,
    Q_inflow_hydrograph=inflow_hydrograph,
    Z_downstream_hydrograph=downstream_hydrograph
)
print("模拟完成。")

## 5. 检查并可视化结果

模拟结束后，我们打印出每个节点的最终水位和流量，并绘制水面线和河床高程。

从结果中可以看到：
- 在闸门所在位置（节点10）的上游，水位被明显抬高，形成了“壅水”或“回水”曲线。这是因为水流受到闸门的阻碍，水位必须升高以获得足够的水头来通过闸门下方的开口。
- 穿过闸门后，水位迅速下降，然后逐渐趋向于正常的流动状态。

In [ ]:
print("\n--- 最终模型状态 ---")
print("节点 | 水位 (m) | 流量 (m^3/s)")
print("---- | --------- | ------------")
for i in range(river.num_nodes):
    print(f"{i:4d} | {river.Z[i]:9.3f} | {river.Q[i]:12.3f}")

# 可视化
fig, ax = plt.subplots(figsize=(12, 6))

node_indices = np.arange(river.num_nodes)
ax.plot(node_indices, river.Z, 'b-o', label='Water Surface Elevation (WSE)')
ax.plot(node_indices, river.Z_bed, 'k-', label='Bed Elevation')

# 标记闸门位置
ax.axvline(gate.node_index, color='r', linestyle='--', label=f'Gate at Node {gate.node_index}')

ax.set_xlabel('Node Index')
ax.set_ylabel('Elevation (m)')
ax.set_title('Final Water Profile with Gate Structure')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()